# Signal Sanity Report — TC-A5

Produces `notebooks/signals_sanity_report.md` covering every `sig_*` column from `utils.signals.add_signals`:
1. Per-signal histogram + time-series plot
2. Pairwise Pearson correlation heatmap
3. Per-signal single-indicator backtest (Sharpe, max drawdown, trade count)

Data: **BTC/USDT 4h, 2020-01-01 ~ 2024-06-30** (training split per design §5.5).

Backtest rule (design doc §4.3): `sig > 0.3` → full long, `sig < -0.3` → flat, otherwise hold previous position. Long-only, no leverage.

**Note**: Volatility-group signals (`sig_vol_*`) express *state*, not direction — their backtest numbers are reported for completeness but should not drive elimination decisions in TC-A7.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.indicators import add_indicators
from utils.signals import add_signals, signal_columns, SIGNAL_WARMUP_WINDOW

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
print("Repo root:", REPO_ROOT)

Repo root: /home/davidlyu/projects/GinkgoBrain


In [2]:
# ── Config ─────────────────────────────────────────────────────────────
SYMBOL      = "BTC/USDT"
TIMEFRAME   = "4h"              # lowercase for pandas >= 2.2
START_DATE  = "2020-01-01"
END_DATE    = "2024-07-01"     # exclusive
PERIODS_PER_YEAR = 6 * 365      # 4h → 6 bars/day

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "signal_sanity_artifacts"
CACHE_DIR    = REPO_ROOT / "data" / "cache"
REPORT_PATH  = REPO_ROOT / "notebooks" / "signals_sanity_report.md"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SYMBOL_SLUG = SYMBOL.replace("/", "")
CACHE_FILE = CACHE_DIR / f"{SYMBOL_SLUG}_{TIMEFRAME}_{START_DATE}_{END_DATE}.parquet"

In [3]:
# ── Load OHLCV (DB → resample to 4h → cache) ───────────────────────────
def load_4h() -> pd.DataFrame:
    if CACHE_FILE.exists():
        print(f"Loading cached 4h OHLCV from {CACHE_FILE}")
        return pd.read_parquet(CACHE_FILE)

    from utils.db import read_ohlcv
    from agents.ppo_shared import resample_ohlcv

    print(f"Fetching 1m OHLCV from DB ({START_DATE} → {END_DATE}) — may take a while.")
    raw = read_ohlcv(SYMBOL, start=START_DATE, end=END_DATE)
    print(f"  1m rows: {len(raw):,}")
    tf = resample_ohlcv(raw, TIMEFRAME)
    print(f"  4h rows: {len(tf):,}")
    tf.to_parquet(CACHE_FILE, index=False)
    return tf

df_raw = load_4h()
df_raw.head()

Loading cached 4h OHLCV from /home/davidlyu/projects/GinkgoBrain/data/cache/BTCUSDT_4h_2020-01-01_2024-07-01.parquet


,timestamp,open,high,low,close,volume
0,2021-01-01 00:00:00+00:00,28923.63,29470.00,28690.17,29278.40,11560.456553
1,2021-01-01 04:00:00+00:00,29278.41,29395.00,28806.54,29092.83,7308.910274
2,2021-01-01 08:00:00+00:00,29092.84,29402.57,28872.24,29313.49,8283.705319
3,2021-01-01 12:00:00+00:00,29313.49,29600.00,29030.14,29188.67,11794.949515
4,2021-01-01 16:00:00+00:00,29188.67,29360.00,28624.57,29029.04,9850.965345


In [4]:
# ── Compute indicators + signals ───────────────────────────────────────
df_ind = add_indicators(df_raw)
df = add_signals(df_ind)
sig_cols = signal_columns(df)
print(f"Rows after indicators: {len(df):,}")
print(f"Signals ({len(sig_cols)}): {sig_cols}")
assert len(sig_cols) == 15, "Expected 15 signals per design §4.2"
assert df[sig_cols].abs().max().max() <= 1.0 + 1e-9, "Signal out of [-1, 1]"

Rows after indicators: 7,627
Signals (15): ['sig_trend_ema_cross', 'sig_trend_macd_hist', 'sig_trend_price_above_ma', 'sig_trend_slope_21', 'sig_mom_rsi_zone', 'sig_mom_rsi_trend', 'sig_mom_stoch_state', 'sig_mom_roc_zscore', 'sig_vol_atr_pct', 'sig_vol_bb_width', 'sig_vol_bb_position', 'sig_volume_obv_slope', 'sig_volume_ratio', 'sig_regime_drawdown', 'sig_regime_return_60']


In [5]:
# ── Drop warmup rows for the analysis (signals == 0 there is meaningless) ─
df_post = df.iloc[SIGNAL_WARMUP_WINDOW:].reset_index(drop=True)
print(f"Post-warmup rows: {len(df_post):,} (dropped first {SIGNAL_WARMUP_WINDOW})")
df_post[sig_cols].describe().T[["mean", "std", "min", "max"]]

Post-warmup rows: 7,527 (dropped first 100)


,mean,std,min,max
sig_trend_ema_cross,-0.001414,0.391214,-1.000000,1.000000
sig_trend_macd_hist,-0.000449,0.351896,-1.000000,1.000000
sig_trend_price_above_ma,0.025371,0.431843,-1.000000,1.000000
sig_trend_slope_21,-0.000602,0.374901,-1.000000,1.000000
sig_mom_rsi_zone,-0.024233,0.364612,-0.800000,0.800000
sig_mom_rsi_trend,0.017657,0.260884,-0.892619,0.852043
sig_mom_stoch_state,0.000133,0.642109,-1.000000,1.000000
sig_mom_roc_zscore,-0.000691,0.336068,-1.000000,1.000000
sig_vol_atr_pct,-0.103760,0.637275,-0.980000,1.000000
sig_vol_bb_width,-0.030392,0.608813,-0.980000,1.000000


## Histograms

In [6]:
def save_histogram(col: str, series: pd.Series, out: Path):
    fig, ax = plt.subplots(figsize=(4.5, 2.8))
    ax.hist(series.dropna(), bins=50, color="#3b7dd8", edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(f"{col} — histogram")
    ax.set_xlim(-1.05, 1.05)
    fig.tight_layout()
    fig.savefig(out, dpi=90)
    plt.close(fig)

for col in sig_cols:
    save_histogram(col, df_post[col], ARTIFACT_DIR / f"hist_{col}.png")
print(f"Wrote {len(sig_cols)} histograms to {ARTIFACT_DIR}")

Wrote 15 histograms to /home/davidlyu/projects/GinkgoBrain/notebooks/signal_sanity_artifacts


## Time-series plots

In [7]:
def save_timeseries(col: str, series: pd.Series, ts: pd.Series, out: Path):
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.plot(ts, series, color="#3b7dd8", linewidth=0.6)
    ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(f"{col} — time series")
    ax.set_ylim(-1.1, 1.1)
    fig.tight_layout()
    fig.savefig(out, dpi=90)
    plt.close(fig)

ts = df_post["timestamp"] if "timestamp" in df_post.columns else df_post.index
for col in sig_cols:
    save_timeseries(col, df_post[col], ts, ARTIFACT_DIR / f"ts_{col}.png")
print(f"Wrote {len(sig_cols)} time-series plots to {ARTIFACT_DIR}")

Wrote 15 time-series plots to /home/davidlyu/projects/GinkgoBrain/notebooks/signal_sanity_artifacts


## Pairwise correlation heatmap

In [8]:
corr = df_post[sig_cols].corr(method="pearson")

fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sig_cols)))
ax.set_yticks(range(len(sig_cols)))
ax.set_xticklabels(sig_cols, rotation=75, ha="right", fontsize=8)
ax.set_yticklabels(sig_cols, fontsize=8)
for i in range(len(sig_cols)):
    for j in range(len(sig_cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                fontsize=6, color="black" if abs(corr.values[i, j]) < 0.6 else "white")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Pairwise Pearson correlation of sig_* columns")
fig.tight_layout()
corr_path = ARTIFACT_DIR / "correlation_heatmap.png"
fig.savefig(corr_path, dpi=110)
plt.close(fig)

# High-correlation pairs for TC-A7 review
pairs = []
for i, a in enumerate(sig_cols):
    for b in sig_cols[i + 1:]:
        pairs.append((a, b, corr.loc[a, b]))
pairs_df = pd.DataFrame(pairs, columns=["a", "b", "pearson"]).sort_values("pearson", key=abs, ascending=False)
high_corr = pairs_df[pairs_df["pearson"].abs() > 0.95].reset_index(drop=True)
print(f"Pairs with |ρ| > 0.95 ({len(high_corr)}):")
print(high_corr.to_string(index=False))

Pairs with |ρ| > 0.95 (3):
                       a                  b   pearson
sig_trend_price_above_ma  sig_mom_rsi_trend  0.981765
     sig_trend_ema_cross sig_trend_slope_21  0.978011
        sig_mom_rsi_zone  sig_mom_rsi_trend -0.951788


## Single-signal backtests

Per design §4.3, each signal drives a naive long-only strategy: `pos = 1` when `sig > 0.3`, `pos = 0` when `sig < -0.3`, else hold previous. Signal values are shifted by 1 bar to avoid look-ahead (decision made on bar-close, executed next bar).

In [9]:
def backtest_signal(sig: pd.Series, ret: pd.Series) -> dict:
    """Return Sharpe (annualized), max drawdown, trade count for one signal."""
    sig_prev = sig.shift(1).fillna(0.0)
    pos = pd.Series(0.0, index=sig.index)
    current = 0.0
    for i, v in enumerate(sig_prev.values):
        if v > 0.3:
            current = 1.0
        elif v < -0.3:
            current = 0.0
        pos.iloc[i] = current
    strat_ret = pos * ret
    mu, sd = strat_ret.mean(), strat_ret.std()
    sharpe = float(mu / sd * np.sqrt(PERIODS_PER_YEAR)) if sd > 1e-12 else 0.0
    equity = (1.0 + strat_ret).cumprod()
    dd = equity / equity.cummax() - 1.0
    mdd = float(dd.min())
    trades = int((pos.diff().fillna(0.0) != 0.0).sum())
    return {"sharpe": sharpe, "max_drawdown": mdd, "total_trades": trades}

log_ret = np.log(df_post["close"] / df_post["close"].shift(1)).fillna(0.0)

bt_rows = []
for col in sig_cols:
    r = backtest_signal(df_post[col], log_ret)
    r["signal"] = col
    bt_rows.append(r)
bt = pd.DataFrame(bt_rows)[["signal", "sharpe", "max_drawdown", "total_trades"]]
bt = bt.sort_values("sharpe", ascending=False).reset_index(drop=True)

# Buy & hold reference
bh_ret = log_ret
bh_sharpe = float(bh_ret.mean() / bh_ret.std() * np.sqrt(PERIODS_PER_YEAR))
bh_equity = (1.0 + bh_ret).cumprod()
bh_mdd = float((bh_equity / bh_equity.cummax() - 1.0).min())
print(f"Buy & Hold: Sharpe={bh_sharpe:.3f}, MaxDD={bh_mdd:.3f}")
bt

Buy & Hold: Sharpe=0.292, MaxDD=-0.840


,signal,sharpe,max_drawdown,total_trades
0,sig_trend_slope_21,0.744661,-0.477688,141
1,sig_mom_rsi_trend,0.692667,-0.509813,98
2,sig_trend_ema_cross,0.682398,-0.544000,117
3,sig_trend_price_above_ma,0.527878,-0.639721,230
4,sig_volume_ratio,0.459959,-0.657024,483
5,sig_regime_return_60,0.390750,-0.683989,100
6,sig_vol_bb_width,0.360103,-0.695838,292
7,sig_vol_bb_position,0.310243,-0.688916,457
8,sig_vol_atr_pct,0.241217,-0.703608,204
9,sig_volume_obv_slope,0.110692,-0.728165,413


## Generate `signals_sanity_report.md`

In [10]:
VOL_SIGNALS = {"sig_vol_atr_pct", "sig_vol_bb_width", "sig_vol_bb_position"}

def rel(p: Path) -> str:
    return str(p.relative_to(REPORT_PATH.parent)).replace("\\", "/")

lines: list[str] = []
lines.append("# Signal Sanity Report\n")
lines.append(f"- Symbol: `{SYMBOL}`  ")
lines.append(f"- Timeframe: `{TIMEFRAME}`  ")
lines.append(f"- Window: `{START_DATE}` → `{END_DATE}` (exclusive)  ")
lines.append(f"- Rows after indicators + signals: `{len(df):,}`, after dropping {SIGNAL_WARMUP_WINDOW}-bar warmup: `{len(df_post):,}`  ")
lines.append(f"- Signals evaluated: **{len(sig_cols)}**\n")
lines.append(f"Generated by `notebooks/signal_sanity.ipynb` — corresponds to TC-A5.\n")

lines.append("## 1. Per-signal distributions and time series\n")
lines.append("| Signal | Group | Histogram | Time series |")
lines.append("|--------|-------|-----------|-------------|")
for col in sig_cols:
    group = col.split("_")[1]
    h = rel(ARTIFACT_DIR / f"hist_{col}.png")
    t = rel(ARTIFACT_DIR / f"ts_{col}.png")
    note = " *(state, not direction)*" if col in VOL_SIGNALS else ""
    lines.append(f"| `{col}`{note} | {group} | ![hist]({h}) | ![ts]({t}) |")
lines.append("")

lines.append("## 2. Pairwise Pearson correlation\n")
lines.append(f"![correlation]({rel(ARTIFACT_DIR / 'correlation_heatmap.png')})\n")
if len(high_corr):
    lines.append(f"### Pairs with |ρ| > 0.95 — TC-A7 candidates for elimination\n")
    lines.append("| Signal A | Signal B | Pearson |")
    lines.append("|----------|----------|---------|")
    for _, row in high_corr.iterrows():
        lines.append(f"| `{row['a']}` | `{row['b']}` | {row['pearson']:+.3f} |")
else:
    lines.append("_No pairs with |ρ| > 0.95._\n")
lines.append("")

lines.append("## 3. Single-signal backtest (long-only, design §4.3)\n")
lines.append(f"Reference — Buy & Hold: Sharpe = **{bh_sharpe:+.3f}**, MaxDD = **{bh_mdd:+.3f}**\n")
lines.append("| Signal | Group | Sharpe | Max Drawdown | Total Trades | Note |")
lines.append("|--------|-------|--------|--------------|--------------|------|")
for _, row in bt.iterrows():
    col = row["signal"]
    group = col.split("_")[1]
    flags = []
    if col in VOL_SIGNALS:
        flags.append("state signal — direction backtest not meaningful")
    elif row["sharpe"] < -0.2:
        flags.append("TC-A7: Sharpe < -0.2, elimination candidate")
    note = "; ".join(flags)
    lines.append(f"| `{col}` | {group} | {row['sharpe']:+.3f} | {row['max_drawdown']:+.3f} | {row['total_trades']} | {note} |")
lines.append("")

REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {REPORT_PATH} ({REPORT_PATH.stat().st_size:,} bytes)")

Wrote /home/davidlyu/projects/GinkgoBrain/notebooks/signals_sanity_report.md (4,598 bytes)
